# Component 2 — Procurement (Buy now / Wait): full pipeline

One notebook from data to research result. It joins (kept unchanged in `archive/`): `procument.ipynb` (original),
`procument_timesplit.ipynb` (time-based evaluation) and `procument_savings_simulation.ipynb` (cost simulation).

**Research question.** Can past market prices tell a shop owner whether to buy an item now or wait,
and how much money does that advice save compared with what shops do today and with simple rules?

| Part | What it does |
|---|---|
| A | Data: load, order in time, what the label means |
| B | Features, time split with a 4-week gap, time-based cross-validation |
| C | Why the original random split over-states the result |
| D | Compare five models on time-CV, pick the champion, test once |
| E | ML vs simple rules on the same test weeks |
| F | Price forecast (regression) vs "price stays the same" |
| G | Money saved: cost simulation (+ bootstrap by week) |
| H | Training/serving check |
| I | Save the model for the app |

**Data safety:** `from_rice_veg_2023_2026.csv` is only read.

In [ ]:
import warnings
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, mean_absolute_error, r2_score, roc_auc_score
from sklearn.model_selection import cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier

warnings.filterwarnings('ignore')
RANDOM_STATE = 42

# Part A — Data

## A1. Load and order in time

In [ ]:
df = pd.read_csv('from_rice_veg_2023_2026.csv').drop_duplicates().reset_index(drop=True)
df['week_start'] = pd.to_datetime(
    df['iso_year'].astype(str) + '-W' + df['iso_week'].astype(str).str.zfill(2) + '-1', format='%G-W%V-%u')
df = df.sort_values(['week_start', 'item']).reset_index(drop=True)
print(f"{len(df)} rows | {df['item'].nunique()} items | {df['week_start'].nunique()} weeks | "
      f"{df['week_start'].min().date()} -> {df['week_start'].max().date()}")
print('Rows per year:', df['iso_year'].value_counts().sort_index().to_dict())
print('Categories:', df['category'].unique().tolist())
print(f"Buy-now rate: {df['target_buy_now'].mean():.3f}")

**Data note:** 2025 has only two weeks (week 1 and week 35), so there is a gap of about seven months.
The prices come from public market price data (rice and vegetables).

## A2. What the label means

In [ ]:
rise = df['expected_price_next_4wk_rs'] / df['current_price_rs'] - 1
for th in [0.0, 0.02, 0.05]:
    agree = ((rise > th).astype(int) == df['target_buy_now']).mean()
    print(f'target_buy_now == (price in 4 weeks > today x (1 + {th:.2f})): {agree:.3f}')

`target_buy_now` = 1 exactly when the price 4 weeks later is higher than today — any rise, even one rupee.
`expected_price_next_4wk_rs` is that future price, so it is **never** given to the model as an input.

# Part B — Features and the time split

Inputs: item, category, ISO year and week, today's price, the price change over the last 4 weeks,
today's price compared with the 3-month average, days to the festival, festival season.

In real use the model only knows the past, so the **test set is the future** (2025 onwards). The label
looks 4 weeks ahead, so a **4-week gap** separates training from test — otherwise the last training
labels would contain test-period prices.

In [ ]:
X = df.drop(columns=['target_buy_now', 'expected_price_next_4wk_rs', 'week_start'])
y_cls = df['target_buy_now'].astype(int)
y_reg = df['expected_price_next_4wk_rs'].astype(float)
cat_cols = ['item', 'category']
num_cols = [c for c in X.columns if c not in cat_cols]
print('Inputs:', list(X.columns))

def make_preprocessor():
    return ColumnTransformer([
        ('num', Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', StandardScaler())]), num_cols),
        ('cat', Pipeline([('imp', SimpleImputer(strategy='most_frequent')), ('ohe', OneHotEncoder(handle_unknown='ignore'))]), cat_cols),
    ])

GAP = pd.Timedelta(weeks=4)
TEST_START = pd.Timestamp('2025-01-01')
dev = (df['week_start'] < TEST_START - GAP).values
test = (df['week_start'] >= TEST_START).values
X_dev, y_dev_cls, y_dev_reg = X[dev], y_cls[dev], y_reg[dev]
X_test, y_test_cls, y_test_reg = X[test], y_cls[test], y_reg[test]
print(f"Development: {dev.sum()} rows -> {df.loc[dev, 'week_start'].max().date()} | gap dropped: {(~dev & ~test).sum()} | "
      f"test: {test.sum()} rows {df.loc[test, 'week_start'].min().date()} -> {df.loc[test, 'week_start'].max().date()}")

## B2. Expanding-window cross-validation (always train on the past, validate on the next block)

In [ ]:
dev_weeks = df.loc[dev, 'week_start'].reset_index(drop=True)
cuts = pd.date_range(dev_weeks.min() + pd.Timedelta(weeks=40), dev_weeks.max(), periods=5)
time_folds = []
for i, (a, b) in enumerate(zip(cuts[:-1], cuts[1:]), 1):
    tr = np.where(dev_weeks < a - GAP)[0]
    va = np.where((dev_weeks >= a) & (dev_weeks < b))[0]
    time_folds.append((tr, va))
    print(f'Fold {i}: train up to {dev_weeks.iloc[tr].max().date()} ({len(tr)} rows) | '
          f'validate {dev_weeks.iloc[va].min().date()} -> {dev_weeks.iloc[va].max().date()} ({len(va)} rows)')

# Part C — Why the original random split over-states the result

`procument.ipynb` shuffled all rows and split 80/20. Neighbouring weeks of the same item then sit in both
train and test, so the model is tested on periods it has effectively seen.

In [ ]:
estimators = {
    'LogisticRegression': LogisticRegression(max_iter=1000, class_weight='balanced', random_state=RANDOM_STATE),
    'DecisionTree': DecisionTreeClassifier(max_depth=6, min_samples_leaf=5, class_weight='balanced', random_state=RANDOM_STATE),
    'RandomForest': RandomForestClassifier(n_estimators=300, max_depth=10, class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1),
    'GradientBoosting': GradientBoostingClassifier(n_estimators=250, learning_rate=0.05, max_depth=4, subsample=0.85, random_state=RANDOM_STATE),
    'XGBoost': XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.05, subsample=0.85, colsample_bytree=0.85,
                             eval_metric='logloss', random_state=RANDOM_STATE, n_jobs=-1),
}
def cls_pipe(name): return Pipeline([('prep', make_preprocessor()), ('clf', clone(estimators[name]))])
reg_pipe = lambda: Pipeline([('prep', make_preprocessor()),
                             ('reg', RandomForestRegressor(n_estimators=200, max_depth=8, random_state=RANDOM_STATE, n_jobs=-1))])

X_tr, X_te, yc_tr, yc_te, yr_tr, yr_te = train_test_split(X, y_cls, y_reg, test_size=0.20, random_state=RANDOM_STATE, stratify=y_cls)
rand_p = cls_pipe('XGBoost').fit(X_tr, yc_tr).predict_proba(X_te)[:, 1]
rand_reg = reg_pipe().fit(X_tr, yr_tr).predict(X_te)
random_split = {'Buy/Wait ROC-AUC': roc_auc_score(yc_te, rand_p), 'Buy/Wait accuracy': accuracy_score(yc_te, rand_p >= .5),
                'Price MAE (LKR)': mean_absolute_error(yr_te, rand_reg), 'Price R²': r2_score(yr_te, rand_reg)}
print('Random split (original method, XGBoost):', {k: round(v, 3) for k, v in random_split.items()})

# Part D — Model comparison on time-based CV

In [ ]:
cv_rows = {}
for name in estimators:
    s = cross_validate(cls_pipe(name), X_dev, y_dev_cls, cv=time_folds, scoring=['roc_auc', 'accuracy', 'f1'], n_jobs=-1)
    cv_rows[name] = {'time-CV ROC-AUC': s['test_roc_auc'].mean(), '± std': s['test_roc_auc'].std(),
                     'accuracy': s['test_accuracy'].mean(), 'F1': s['test_f1'].mean()}
cv_table = pd.DataFrame(cv_rows).T.round(3)
best_name = cv_table['time-CV ROC-AUC'].idxmax()
print(f'Champion (validation folds only): {best_name}')
cv_table

## D2. Test once on the future period

In [ ]:
champion = cls_pipe(best_name).fit(X_dev, y_dev_cls)
proba = champion.predict_proba(X_test)[:, 1]
test_auc, test_acc, test_f1 = roc_auc_score(y_test_cls, proba), accuracy_score(y_test_cls, proba >= .5), f1_score(y_test_cls, proba >= .5)
print(f'{best_name} on the test period: ROC-AUC {test_auc:.3f} | accuracy {test_acc:.3f} | F1 {test_f1:.3f}')
print(f"Random split said ROC-AUC {random_split['Buy/Wait ROC-AUC']:.3f}  ->  over-stated by "
      f"{random_split['Buy/Wait ROC-AUC'] - test_auc:+.3f}")

# Part E — ML vs simple rules on the same test weeks

In [ ]:
majority = int(y_dev_cls.mean() >= 0.5)
fell = -X_test['price_change_4wk_pct'].fillna(0)
below = -X_test['price_vs_3mo_avg_pct'].fillna(0)
rows = {
    f"Always {'BUY' if majority else 'WAIT'}": (0.5, accuracy_score(y_test_cls, np.full(len(y_test_cls), majority)),
                                                  f1_score(y_test_cls, np.full(len(y_test_cls), majority))),
    'Rule: buy if price fell (4 wk)': (roc_auc_score(y_test_cls, fell), accuracy_score(y_test_cls, fell > 0), f1_score(y_test_cls, fell > 0)),
    'Rule: buy if below 3-mo average': (roc_auc_score(y_test_cls, below), accuracy_score(y_test_cls, below > 0), f1_score(y_test_cls, below > 0)),
    f'ML: {best_name}': (test_auc, test_acc, test_f1),
}
baselines = pd.DataFrame(rows, index=['ROC-AUC', 'Accuracy', 'F1']).T.round(3)
baselines

Prices here tend to **revert**: after a fall they usually rise again. A one-line rule ("buy if the price
fell over the last 4 weeks") already comes close to the ML model. Part G checks whether the difference
matters in money.

# Part F — Price forecast: model vs "price stays the same"

In [ ]:
price_regressor = reg_pipe().fit(X_dev, y_dev_reg)
price_pred = price_regressor.predict(X_test)
naive = X_test['current_price_rs']
reg_mae, reg_r2 = mean_absolute_error(y_test_reg, price_pred), r2_score(y_test_reg, price_pred)
naive_mae = mean_absolute_error(y_test_reg, naive)
print(f'RandomForest price forecast : MAE LKR {reg_mae:6.2f} | R² {reg_r2:.3f}')
print(f'Naive "price stays the same": MAE LKR {naive_mae:6.2f} | R² {r2_score(y_test_reg, naive):.3f}')
print(f'MAE improvement over naive  : {(1 - reg_mae / naive_mae) * 100:.1f}%')

R² is high for both because prices change slowly; the **MAE improvement over naive** is the meaningful number.

# Part G — Does the advice save money?

For every item and test week: **buy now** → pay today's price; **wait** → pay the actual price 4 weeks later.
One unit per decision. Reference = *always buy now* (what a shop does without advice); upper bound =
*perfect foresight*.

In [ ]:
T = df[test].copy()
T['p_rise'] = proba
T['forecast_4wk'] = price_pred
now, later = T['current_price_rs'], T['expected_price_next_4wk_rs']
policies = {
    '0. Always buy now (today)': np.ones(len(T), dtype=bool),
    '1. Rule: price fell (4 wk)': (T['price_change_4wk_pct'].fillna(0) < 0).to_numpy(),
    '2. Rule: below 3-mo average': (T['price_vs_3mo_avg_pct'].fillna(0) < 0).to_numpy(),
    '3. ML classifier (used by the app)': (T['p_rise'] >= 0.5).to_numpy(),
    '4. ML price forecast': (T['forecast_4wk'] > now).to_numpy(),
    '★ Perfect foresight': (later > now).to_numpy(),
}
cost = {k: np.where(b, now, later) for k, b in policies.items()}
ref = cost['0. Always buy now (today)'].sum(); best = ref - cost['★ Perfect foresight'].sum()
savings = pd.DataFrame([{'policy': k, 'total spend (LKR)': c.sum(), 'saved (LKR)': ref - c.sum(),
                         'saved %': (ref - c.sum()) / ref * 100, '% of possible saving': (ref - c.sum()) / best * 100}
                        for k, c in cost.items()]).set_index('policy')
print(f'{len(T)} decisions, 1 unit each; spend if always buying now: LKR {ref:,.0f}')
savings.round(2)

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
weeks = T['week_start'].to_numpy(); uniq = np.unique(weeks)
week_rows = [np.where(weeks == w)[0] for w in uniq]
saved_rows = {k: cost['0. Always buy now (today)'] - c for k, c in cost.items()}
def boot_pct(k):
    out = []
    for _ in range(1000):
        pick = np.concatenate([week_rows[i] for i in rng.integers(0, len(uniq), len(uniq))])
        out.append(saved_rows[k][pick].sum() / cost['0. Always buy now (today)'][pick].sum() * 100)
    return np.array(out)
boot = {k: boot_pct(k) for k in ['1. Rule: price fell (4 wk)', '3. ML classifier (used by the app)']}
for k, b in boot.items():
    lo, hi = np.percentile(b, [2.5, 97.5]); print(f'{k:36s} saving {b.mean():5.2f}%  95% CI [{lo:5.2f}, {hi:5.2f}]')
d = boot['3. ML classifier (used by the app)'] - boot['1. Rule: price fell (4 wk)']
lo, hi = np.percentile(d, [2.5, 97.5])
print(f'\nML minus rule: {d.mean():+.2f} percentage points  95% CI [{lo:+.2f}, {hi:+.2f}]')

Whole **weeks** are resampled because decisions in the same week share the same market conditions.

In [ ]:
by_cat = pd.DataFrame({'category': T['category'].to_numpy(), 'spend': cost['0. Always buy now (today)'],
                       'rule': saved_rows['1. Rule: price fell (4 wk)'], 'ML': saved_rows['3. ML classifier (used by the app)'],
                       'perfect': saved_rows['★ Perfect foresight']}).groupby('category').sum()
for c in ['rule', 'ML', 'perfect']:
    by_cat[f'saved % ({c})'] = by_cat[c] / by_cat['spend'] * 100
by_cat[[c for c in by_cat.columns if c.startswith('saved %')]].round(2)

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(18, 4.8))
names = list(cv_table.index)
ax[0].barh(names, cv_table['time-CV ROC-AUC'], xerr=cv_table['± std'], color='#6a9fd4')
ax[0].set_xlim(0.5, 1.0); ax[0].set_xlabel('Time-CV ROC-AUC'); ax[0].set_title('Model comparison (time-based CV)')
ax[1].scatter(y_test_reg, price_pred, alpha=0.35, s=10, color='#e08e79')
lims = [y_test_reg.min(), y_test_reg.max()]; ax[1].plot(lims, lims, 'k--', lw=1)
ax[1].set_xlabel('Actual price in 4 weeks (LKR)'); ax[1].set_ylabel('Predicted'); ax[1].set_title('Price forecast (test period)')
s = savings.drop(index='0. Always buy now (today)')
ax[2].barh(s.index, s['saved %'], color=['#999', '#999', '#1f5fa8', '#6a9fd4', '#2e7d32'])
for i, v in enumerate(s['saved %']): ax[2].text(v + 0.03, i, f'{v:.2f}%', va='center', fontsize=9)
ax[2].invert_yaxis(); ax[2].set_xlabel('Saving vs always buying now (%)'); ax[2].set_title('Money saved (test period)')
plt.tight_layout(); plt.savefig('procurement_buy_wait_pipeline.png', dpi=120); plt.show()

# Part H — Training/serving check

The app builds the inputs in `backend/src/utils/features.js` (`buildProcurementFeatures`, `priceTrendFeatures`):

- `price_change_4wk_pct` and `price_vs_3mo_avg_pct` are computed from the **shop's own purchase prices**,
  with the same definitions as this dataset (earlier the backend always sent 0 — fixed).
- If an item has no purchase history the app sends nothing and the model fills the training median.
  Below: how much the decision depends on these two inputs.

In [ ]:
X_no_history = X_test.copy()
X_no_history[['price_change_4wk_pct', 'price_vs_3mo_avg_pct']] = np.nan
p_no_hist = champion.predict_proba(X_no_history)[:, 1]
print(f'Test ROC-AUC with price history   : {test_auc:.3f}')
print(f'Test ROC-AUC without price history: {roc_auc_score(y_test_cls, p_no_hist):.3f}')

Without its own purchase history the test ROC-AUC drops from about 0.80 to about 0.58 — barely better than a
coin toss. The price trend is what the model uses. In the app the Buy/Wait decision itself comes from stock and the demand forecast; this model adds
the **price context** ("good price now" / "prices may improve").

# Part I — Save the model for the app

Refit the champion and the price regressor on **all** rows (the usual last step: the app should use the
most recent prices), in the bundle format the ML service reads. A copy of this file is the app's model
(`ml_service/models/procurement_buy_wait_model.pkl`); it replaced the original random-split model.

In [ ]:
final_cls = cls_pipe(best_name).fit(X, y_cls)
final_reg = reg_pipe().fit(X, y_reg)
bundle = {
    'classifier_model': final_cls,
    'regressor_model': final_reg,
    'feature_names': X.columns.tolist(),
    'numerical_cols': num_cols,
    'categorical_cols': cat_cols,
    'evaluation': {'method': 'time split from 2025-01-01, 4-week gap, expanding-window CV',
                   'champion': best_name, 'test_roc_auc': round(test_auc, 4), 'test_f1': round(test_f1, 4),
                   'price_mae_lkr': round(reg_mae, 2), 'naive_price_mae_lkr': round(naive_mae, 2)},
}
joblib.dump(bundle, 'procurement_buy_wait_model.pkl', compress=3)
print('Saved procurement_buy_wait_model.pkl ->', bundle['evaluation'])

# Conclusion

*Written after reading the numbers above — fill in with the actual results:*

- Honest test (future weeks): ROC-AUC ___ (the original random split said ___).
- ML vs the "price fell" rule: ROC-AUC ___ vs ___; money saved ___ % vs ___ % (difference interval ___).
- Following the advice saves about ___ % of the purchase bill, ___ % of the best possible.

**Limitations.** 2025 is almost missing from the data. The label counts any price rise, however small.
The simulation buys one unit per decision and ignores storage, spoilage and transport. The prices are
public market prices, not one shop's purchases.